
# Actividad práctica — Análisis exploratorio de datos en salud pública chilena

**Curso:** Análisis de Datos e Inferencia Estadística  
**Duración estimada:** ~60 minutos  
**Contexto:** Atención Primaria de Salud (APS) en Chile

## Objetivo
Aplicar herramientas de **análisis exploratorio de datos (EDA)** y **estadística descriptiva** utilizando un dataset clínico inspirado en chequeos preventivos realizados en centros de salud chilenos (CESFAM).

Trabajaremos con variables comunes en programas preventivos como:

- Edad
- Sexo
- IMC
- Presión arterial
- Glucosa en ayunas
- Colesterol total
- Tabaquismo
- Actividad física
- Diagnóstico de hipertensión
- Diagnóstico de diabetes tipo 2


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)


# Generación de dataset clínico simulado (tipo CESFAM)


In [ ]:

n = 200

edad = np.random.randint(20,80,n)

sexo = np.random.choice(["Femenino","Masculino"],n)

actividad = np.random.choice(["Baja","Moderada","Alta"],n,p=[0.4,0.4,0.2])

tabaquismo = np.random.choice(["Si","No"],n,p=[0.3,0.7])

imc = np.random.normal(28,4,n)
imc += np.where(actividad=="Baja",1.5,0)
imc += np.where(actividad=="Alta",-1.2,0)

presion = np.random.normal(120,15,n)
presion += (edad-45)*0.5
presion += (imc-27)*0.7

glucosa = np.random.normal(95,18,n)
glucosa += (imc-27)*1.5

colesterol = np.random.normal(200,35,n)
colesterol += (edad-45)*0.6
colesterol += np.where(tabaquismo=="Si",10,0)

hta = np.where(presion>=140,"Si","No")
dm2 = np.where(glucosa>=126,"Si","No")

score = (
(edad>=55).astype(int)+
(imc>=30).astype(int)+
(presion>=140).astype(int)+
(glucosa>=126).astype(int)+
(colesterol>=240).astype(int)
)

riesgo = pd.cut(score,bins=[-1,1,3,10],labels=["Bajo","Moderado","Alto"])

df = pd.DataFrame({
"Edad":edad,
"Sexo":sexo,
"IMC":np.round(imc,1),
"Presion_sistolica":presion.astype(int),
"Glucosa_ayunas":glucosa.astype(int),
"Colesterol_total":colesterol.astype(int),
"Tabaquismo":tabaquismo,
"Actividad_fisica":actividad,
"Diagnostico_HTA":hta,
"Diagnostico_DM2":dm2,
"Riesgo_cardiovascular":riesgo
})

df.head()



# Exploración inicial


In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.select_dtypes(include="number").columns


### Preguntas

1. ¿Cuántos pacientes hay en la muestra?
2. ¿Qué variables son numéricas?
3. ¿Qué variables son categóricas?



# Frecuencia de diagnósticos


In [ ]:
df["Diagnostico_HTA"].value_counts()

In [ ]:
df["Diagnostico_DM2"].value_counts()

In [ ]:
df.groupby("Diagnostico_HTA")["Edad"].mean()

In [ ]:
df.groupby("Actividad_fisica")["Glucosa_ayunas"].mean()

In [ ]:
pd.crosstab(df["Sexo"], df["Diagnostico_HTA"])


# Medidas de tendencia central


In [ ]:
df[["Edad","IMC","Presion_sistolica","Glucosa_ayunas","Colesterol_total"]].agg(["mean","median","std"])


# Visualización


In [ ]:
df["Presion_sistolica"].hist(bins=15)
plt.title("Distribución presión arterial")
plt.show()

In [ ]:
df["Riesgo_cardiovascular"].value_counts().plot(kind="bar")
plt.title("Riesgo cardiovascular")
plt.show()

In [ ]:
grupos = [
    df[df["Riesgo_cardiovascular"]=="Bajo"]["Edad"],
    df[df["Riesgo_cardiovascular"]=="Moderado"]["Edad"],
    df[df["Riesgo_cardiovascular"]=="Alto"]["Edad"]
]

plt.boxplot(grupos, labels=["Bajo","Moderado","Alto"])
plt.ylabel("Edad")
plt.title("Edad según riesgo cardiovascular")

plt.show()

In [ ]:
# Relaciones entre variables
plt.scatter(df["IMC"],df["Glucosa_ayunas"])
plt.xlabel("IMC")
plt.ylabel("Glucosa")

In [ ]:
corr = df.corr(numeric_only=True)

plt.imshow(corr)

plt.colorbar()

plt.xticks(range(len(corr.columns)), corr.columns, rotation=45)
plt.yticks(range(len(corr.columns)), corr.columns)

plt.title("Matriz de correlación")

plt.tight_layout()
plt.show()


# Conclusión

Describe:

- la población estudiada
- un hallazgo relevante
- una posible implicancia en salud pública
